# Tutorial: Beamstop Definition

This notebook isolates the detector and beamstop part of the scattering notebook.

The beamstop is a detector-plane mask. In the code, the user gives the physical beamstop radius at the beamstop plane, and the simulator projects it to the detector plane using the sample-detector and detector-beamstop distances. This makes the mask sensitive to geometry, not just to pixel size.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    %matplotlib widget
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator


## 1. Define detector geometry

Start with a modest detector so the visualizations stay responsive. The center is given in pixel coordinates as `(row, column)`.


In [ ]:
detector_shape = (512, 512)
detector_center = tuple(np.array(detector_shape) // 2)

# Detector pixels and distances are physical quantities.
detector_pixel_size = 20e-6          # m / detector pixel
sample_to_detector_distance = 0.075  # m

# The beam config is only needed here to compute q-space and real-space resolution.
xray_config = sim.XRayConfig(
    energy=778.0,                    # eV
    photon_flux=1e10,                # photons / s
    pol="CR",
    coherence_length=(10e-6, 10e-6),
)
xray_config.setup()


## 2. Create a circular beamstop

Important user-facing parameters:

- `radius`: physical beamstop radius in metres at the beamstop plane.
- `sigma`: Gaussian smoothing of the mask edge, also in metres.
- `wire_width`: support-wire width in metres. Set to `0` for no wire.
- `angle`: beamstop ellipse and wire angle in radians.
- `ellipticity`: `(min, max)` range. `(1, 1)` gives a circle.
- `roughness`: random boundary modulation amplitude.
- `roughness_modes`: angular Fourier modes used for roughness.
- `antialias`: supersampling factor for smoother thin wires and diagonal edges.
- `seed`: makes the random angle/roughness reproducible.


In [ ]:
beamstop_config = sim.BeamstopConfig(
    bs_method="circular",
    bs_detector_distance=0.010,      # m between detector and beamstop plane
    bs_center=detector_center,
    bs_config={
        "radius": 250e-6,            # m
        "sigma": 15e-6,              # m
        "angle": np.deg2rad(18),
        "ellipticity": (1.0, 1.25),
        "roughness": 0.04,
        "roughness_modes": (3, 12),
        "wire_width": 55e-6,
        "wire_bend": 30e-6,
        "antialias": 4,
        "seed": 7,
    },
)

detector_config = sim.DetectorConfig(
    shape=detector_shape,
    pixel_size=detector_pixel_size,
    sample_to_detector_distance=sample_to_detector_distance,
    detector_center=detector_center,
    beamstop_config=beamstop_config,
)
detector_config.setup()
real_space_resolution = detector_config.calc_realspace_resolution(xray_config.beam_params)

print(f"Detector shape: {detector_shape}")
print(f"Beamstop mask range: {detector_config.detector_layout.beamstop.min():.3f} to {detector_config.detector_layout.beamstop.max():.3f}")
print(f"Real-space detector resolution: {real_space_resolution * 1e9:.2f} nm")


## 3. Visualize the mask in pixels and millimetres

In [ ]:
detector_config.visualize_beamstop()


## 4. Inspect a central line profile

The profile reveals edge smoothing, antialiasing, and wire intersections better than an image alone.


In [ ]:
beamstop_mask = detector_config.detector_layout.beamstop
row = detector_center[0]
col = detector_center[1]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(beamstop_mask[row, :])
ax[0].set_title("Horizontal line through beamstop center")
ax[0].set_xlabel("x pixel")
ax[0].set_ylabel("mask value")

ax[1].plot(beamstop_mask[:, col])
ax[1].set_title("Vertical line through beamstop center")
ax[1].set_xlabel("y pixel")
ax[1].set_ylabel("mask value")


## 5. Compare parameter choices

Use this cell when teaching what each artifact does. Each row changes one aspect while keeping the detector geometry fixed.


In [ ]:
variants = [
    ("clean circle", {"radius": 250e-6, "sigma": 0.0, "wire_width": 0.0, "seed": 1}),
    ("soft edge", {"radius": 250e-6, "sigma": 25e-6, "wire_width": 0.0, "seed": 1}),
    ("with wire", {"radius": 250e-6, "sigma": 10e-6, "wire_width": 55e-6, "angle": np.deg2rad(35), "seed": 1}),
    ("rough ellipse", {"radius": 250e-6, "sigma": 10e-6, "ellipticity": (1.2, 1.2), "roughness": 0.06, "roughness_modes": (3, 14), "seed": 3}),
]

fig, axes = plt.subplots(1, len(variants), figsize=(13, 3.2), sharex=True, sharey=True)
for ax, (title, cfg) in zip(axes, variants):
    bs_cfg = sim.BeamstopConfig(
        bs_method="circular",
        bs_detector_distance=0.010,
        bs_center=detector_center,
        bs_config={**cfg, "antialias": 4},
    )
    det_cfg = sim.DetectorConfig(
        shape=detector_shape,
        pixel_size=detector_pixel_size,
        sample_to_detector_distance=sample_to_detector_distance,
        detector_center=detector_center,
        beamstop_config=bs_cfg,
    )
    det_cfg.setup()
    ax.imshow(det_cfg.detector_layout.beamstop, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_axis_off()
